# Ensemble Seperation

### In this notebook we run statistical tests on how the ensemble activation (Ensemble12) differs between differnt outcomes and choices

In [1]:
import os
import sys
import re
import numpy as np
import pandas as pd
import plotly.express as px
import scipy.stats as stats
from dash import html, dcc, Input, Output, Dash
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append(os.path.join(os.getcwd(), '..', '..', '..'))
from baseVR.base_functionality import init_import_paths
init_import_paths() 

from CustomLogger import CustomLogger as Logger
from analytics_processing import analytics

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args, fullfnames2snames
from dashsrc.plot_components.plots import plot_TrackFiringRate
from dashsrc.plot_components.plots import plot_unit_fr_stability


In [2]:
Logger().init_logger(None, None, logging_level="DEBUG")
animal_ids = [6]
paradigm = [1100]
session_range = [1,33]
session_ids = None
normalize = True
smooth = False
excl_session_names = None  #['2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min', '2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min', '2024-12-11_17-42_rYL006_P1100_LinearTrackStop_30min']

session_dirs = sessionlist_fullfnames_from_args(paradigm, animal_ids, session_ids, excl_session_names=excl_session_names)[0]
session_names= fullfnames2snames(session_dirs)

2026-02-25 15:04:02,414|DEBUG|8090|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Searching NAS for applicable sessions...
2026-02-25 15:04:03,598|DEBUG|8090|sessions_from_nas_parsing|get_sessionlist_fullfnames
	For paradigms [1100], animals [6], found 34 sessions.
2026-02-25 15:04:03,604|DEBUG|8090|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: [1100], animal_ids: [6], session_ids: None, from_date: None, to_date: None
	Merging 34 sessions



In [3]:
# firing rates and behavior data
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
# fr_track = analytics.get_analytics('FiringRateTrackwiseHz', session_names=session_names)
#fr_z_scored  = analytics.get_analytics('FiringRate40msZ', session_names=session_names)
#fr_z_all_sess = fr.apply(lambda unit_fr: ((unit_fr - unit_fr.mean()) / unit_fr.std()))
t0_events = analytics.get_analytics('TrialWiseT0Events40ms', session_names=session_names)
# behav = analytics.get_analytics('BehaviorTrackwise', session_names=session_names)
# behav.index = behav.index.droplevel(('animal_id', 'paradigm_id', 'entry_id'))

# ensamble related data
t0_ens = analytics.get_analytics('EnsembleT0Projection', session_names=session_names)
# ensambles = analytics.get_analytics('ConcatenatedEnsambles40ms', session_names=session_names)
# ens_data = analytics.get_analytics('TrackwiseEnsembleProj', session_names=session_names)
# ensamble_proj = analytics.get_analytics('ConcatenatedEnsambleProj40ms', session_names=session_names)#.drop("to_ephys_timestamp", axis=1)
# ensamble_proj.set_index(['session_id'], inplace = True)
# ens_data = ens_data.set_index(['session_id', 'trial_id']).sort_index()

2026-02-25 15:04:03,617|DEBUG|8090|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-25 15:04:04,028|DEBUG|8090|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 34 sessions

2026-02-25 15:04:04,029|DEBUG|8090|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-25 15:04:04,048|INFO|8090|analytics|get_analytics
	Analytic `FiringRate40msHz` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-02-25 15:04:04,048|DEBUG|8090|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrackStop_21min.hdf5
/Vol

In [4]:
t0_ens

from_ephys_timestamp  \
paradigm_id animal_id session_id       entry_id                         
1100        6         2024-11-14_16-40 0                      4400000   
                                       1                      4440000   
                                       2                      4480000   
                                       3                      4520000   
                                       4                      4560000   
...                                                               ...   
                      2025-01-27_13-39 20515               4399880000   
                                       20516               4399920000   
                                       20517               4399960000   
                                       20518               4400000000   
                                       20519               4400040000   

                                                 to_ephys_timestamp  \
paradigm_id animal_id session_id       entry_id                       
1100        6         2024-11-14_16-40 0                    4440000   
                                       1                    4480000   
                                       2                    4520000   
                                       3                    4560000   
                                       4                    4600000   
...                                                             ...   
                      2025-01-27_13-39 20515             4399920000   
                                       20516             4399960000   
                                       20517             4400000000   
                                       20518             4400040000   
                                       20519             4400080000   

                                                 Assembly001  Assembly002  \
paradigm_id animal_id session_id       entry_id                             
1100        6         2024-11-14_16-40 0            0.044591    -0.079732   
                                       1            0.353522    -0.208656   
                                       2           -0.090020    -0.381483   
                                       3           -0.082682    -0.003392   
                                       4            0.169519    -0.213003   
...                                                      ...          ...   
                      2025-01-27_13-39 20515        0.079890     1.072596   
                                       20516       -0.044071    -0.135663   
                                       20517       -0.154243    -0.233770   
                                       20518       -0.133507    -0.307351   
                                       20519       -0.014874     0.074854   

                                                 Assembly003  Assembly004  \
paradigm_id animal_id session_id       entry_id                             
1100        6         2024-11-14_16-40 0            0.117157    -0.100619   
                                       1           -0.050525    -0.098752   
                                       2            0.185447     1.103287   
                                       3            0.093962    -0.152654   
                                       4           -0.009496    -0.107002   
...                                                      ...          ...   
                      2025-01-27_13-39 20515       -0.127818    -0.427801   
                                       20516       -0.135668    -0.157276   
                                       20517       -0.216997    -0.494509   
                                       20518       -0.067960    -0.653956   
                                       20519        0.177083     1.249869   

                                                 Assembly005  Assembly006  \
paradigm_id animal_id session_id       entry_id                             
1100        6    

# Expert Trials Only


## Session wise pooling
### Comparing session wise and interval specific differences between: 
- Cue1 and Cu2 -> Can we see statistically significant seperation between the different cues? 
- Choice Skip vs. Stop -> Is the ensemble more active when stopping vs. when skipping in a reward zone?
- If there is a difference in choice, can we see it already in the cue zone and do these things realte?

In [5]:
# --- Assembly012: expert-trial interval analysis (trial-level means) ---
TARGET_ASSEMBLY = 'Assembly012'
TARGET_INTERVALS = ['R1_entry_interval', 'R2_entry_interval', 'cue_entry_interval']


def _analytics_df_with_columns(df):
    out = df.copy()
    idx_names = list(out.index.names) if isinstance(out.index, pd.MultiIndex) else [out.index.name]
    needs_index_reset = any((name is not None) and (name not in out.columns) for name in idx_names)
    if needs_index_reset:
        out = out.reset_index()
    else:
        out = out.reset_index(drop=True)
    return out


def prepare_expert_trial_interval_means(t0_ens, assembly_col=TARGET_ASSEMBLY, intervals=TARGET_INTERVALS):
    df = _analytics_df_with_columns(t0_ens)

    df = df[df['interval_name'].isin(intervals)].copy()
    df['cue'] = pd.to_numeric(df['cue'], errors='coerce')
    df = df[df[assembly_col].notna()].copy()

    expert_mask = (
        ((df['cue'] == 1) & df['choice_R1'].eq(True) & df['choice_R2'].eq(False)) |
        ((df['cue'] == 2) & df['choice_R1'].eq(False) & df['choice_R2'].eq(True))
    )
    expert = df.loc[expert_mask].copy()

    expert['cue_label'] = expert['cue'].map({1.0: 'cue1', 2.0: 'cue2'})
    expert['r1_action'] = np.where(expert['choice_R1'].eq(True), 'stop_R1', 'skip_R1')

    group_cols = [
        'session_id', 'trial_id', 'interval_name', 'cue', 'cue_label',
        'choice_R1', 'choice_R2', 'r1_action'
    ]
    trial_means = (
        expert.groupby(group_cols, dropna=False, as_index=False)[assembly_col]
        .mean()
        .rename(columns={assembly_col: 'assembly_mean'})
    )

    trial_means['session_dt'] = pd.to_datetime(
        trial_means['session_id'], format='%Y-%m-%d_%H-%M', errors='coerce'
    )
    session_order_df = (
        trial_means[['session_id', 'session_dt']]
        .drop_duplicates()
        .sort_values(['session_dt', 'session_id'], na_position='last')
        .reset_index(drop=True)
    )
    session_order_df['session_order'] = np.arange(1, len(session_order_df) + 1)
    session_order_map = dict(zip(session_order_df['session_id'], session_order_df['session_order']))
    trial_means['session_order'] = trial_means['session_id'].map(session_order_map)

    return trial_means, session_order_df


def _ttest_result(a, b):
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nan
    res = stats.ttest_ind(a, b, equal_var=False, nan_policy='omit')
    if hasattr(res, 'statistic'):
        return float(res.statistic), float(res.pvalue)
    t_stat, p_val = res
    return float(t_stat), float(p_val)


def _mannwhitney_result(a, b):
    if len(a) < 1 or len(b) < 1:
        return np.nan, np.nan
    try:
        res = stats.mannwhitneyu(a, b, alternative='two-sided')
        if hasattr(res, 'statistic'):
            return float(res.statistic), float(res.pvalue)
        u_stat, p_val = res
        return float(u_stat), float(p_val)
    except Exception:
        return np.nan, np.nan


def compare_two_groups(sub_df, group_col, group_a, group_b, value_col='assembly_mean'):
    a = sub_df.loc[sub_df[group_col] == group_a, value_col].dropna().to_numpy()
    b = sub_df.loc[sub_df[group_col] == group_b, value_col].dropna().to_numpy()

    t_stat, t_p = _ttest_result(a, b)
    u_stat, u_p = _mannwhitney_result(a, b)

    mean_a = float(np.mean(a)) if len(a) else np.nan
    mean_b = float(np.mean(b)) if len(b) else np.nan
    diff_b_minus_a = mean_b - mean_a if len(a) and len(b) else np.nan

    # One-sided p-value for H1: mean(group_b) > mean(group_a)
    if np.isnan(t_stat) or np.isnan(t_p):
        t_p_one_sided_b_gt_a = np.nan
    else:
        t_p_one_sided_b_gt_a = (t_p / 2.0) if (t_stat > 0) else (1.0 - t_p / 2.0)

    return {
        'n_a': int(len(a)),
        'n_b': int(len(b)),
        'mean_a': mean_a,
        'mean_b': mean_b,
        'std_a': float(np.std(a, ddof=1)) if len(a) > 1 else np.nan,
        'std_b': float(np.std(b, ddof=1)) if len(b) > 1 else np.nan,
        'diff_b_minus_a': diff_b_minus_a,
        'welch_t': t_stat,
        'welch_p_two_sided': t_p,
        'welch_p_one_sided_b_gt_a': t_p_one_sided_b_gt_a,
        'mw_u': u_stat,
        'mw_p_two_sided': u_p,
    }


def summarize_overall(trial_means, group_col, group_a, group_b, intervals=TARGET_INTERVALS):
    rows = []
    for interval in intervals:
        sub = trial_means.loc[trial_means['interval_name'] == interval]
        stats_row = compare_two_groups(sub, group_col, group_a, group_b)
        rows.append({
            'interval_name': interval,
            'group_a': group_a,
            'group_b': group_b,
            **stats_row,
        })
    return pd.DataFrame(rows)


def summarize_by_session(trial_means, group_col, group_a, group_b, intervals=TARGET_INTERVALS):
    rows = []
    for (session_id, interval), sub in trial_means.groupby(['session_id', 'interval_name'], sort=False):
        if interval not in intervals:
            continue
        stats_row = compare_two_groups(sub, group_col, group_a, group_b)
        rows.append({
            'session_id': session_id,
            'interval_name': interval,
            'group_a': group_a,
            'group_b': group_b,
            **stats_row,
        })

    out = pd.DataFrame(rows)
    out['session_dt'] = pd.to_datetime(out['session_id'], format='%Y-%m-%d_%H-%M', errors='coerce')
    out = out.sort_values(['session_dt', 'session_id', 'interval_name'], na_position='last').reset_index(drop=True)
    session_order_map = (
        out[['session_id', 'session_dt']]
        .drop_duplicates()
        .sort_values(['session_dt', 'session_id'], na_position='last')
        .reset_index(drop=True)
    )
    session_order_map['session_order'] = np.arange(1, len(session_order_map) + 1)
    out = out.merge(session_order_map[['session_id', 'session_order']], on='session_id', how='left')
    return out


def summarize_trend(session_effects, diff_col='diff_b_minus_a', intervals=TARGET_INTERVALS):
    rows = []
    for interval in intervals:
        sub = session_effects.loc[
            (session_effects['interval_name'] == interval) &
            session_effects[diff_col].notna() &
            session_effects['session_order'].notna()
        ].sort_values('session_order')

        n_sessions = len(sub)
        if n_sessions >= 3:
            rho, p_val = stats.spearmanr(sub['session_order'], sub[diff_col])
        else:
            rho, p_val = (np.nan, np.nan)

        rows.append({
            'interval_name': interval,
            'n_sessions': n_sessions,
            'spearman_rho': float(rho) if pd.notna(rho) else np.nan,
            'spearman_p': float(p_val) if pd.notna(p_val) else np.nan,
        })

    return pd.DataFrame(rows)


trial_means_ens12, session_order_ens12 = prepare_expert_trial_interval_means(t0_ens)

print(f"Expert trial-level means rows: {len(trial_means_ens12)}")
print("Intervals present:", sorted(trial_means_ens12['interval_name'].dropna().unique().tolist()))
print()
print("Trial counts per interval and cue (expert trials):")
print(
    trial_means_ens12.groupby(['interval_name', 'cue_label'])['trial_id']
    .count()
    .unstack(fill_value=0)
    .reindex(TARGET_INTERVALS, fill_value=0)
)






Expert trial-level means rows: 2403
Intervals present: ['R1_entry_interval', 'R2_entry_interval', 'cue_entry_interval']

Trial counts per interval and cue (expert trials):
cue_label           cue1  cue2
interval_name                 
R1_entry_interval    371   430
R2_entry_interval    371   430
cue_entry_interval   371   430


In [6]:
# Overall (pooled expert trials across sessions): cue-only tests + plots

overall_cue1_vs_cue2 = summarize_overall(trial_means_ens12, 'cue_label', 'cue1', 'cue2')

print('Overall comparison (expert trials): cue1 vs cue2 (group_b - group_a = cue2 - cue1)')
display(overall_cue1_vs_cue2)


def plot_overall_group_comparison(trial_means, summary_df, group_col, group_order, title, colors=None, show=True):
    if colors is None:
        colors = ['#f28e2b', '#7b2cbf']

    n_cols = len(TARGET_INTERVALS)
    fig = make_subplots(
        rows=1,
        cols=n_cols,
        shared_yaxes=True,
        subplot_titles=TARGET_INTERVALS,
        horizontal_spacing=0.06,
    )

    summary_map = summary_df.set_index('interval_name')

    for col_idx, interval in enumerate(TARGET_INTERVALS, start=1):
        sub = trial_means.loc[trial_means['interval_name'] == interval].copy()
        for grp, color in zip(group_order, colors):
            vals = sub.loc[sub[group_col] == grp, 'assembly_mean'].dropna().to_numpy()
            fig.add_trace(
                go.Box(
                    y=vals,
                    x=[grp] * len(vals),
                    name=str(grp),
                    legendgroup=str(grp),
                    showlegend=False,
                    marker=dict(color=color, opacity=0.35, size=5),
                    line=dict(color=color),
                    fillcolor=f"rgba{(*px.colors.hex_to_rgb(color), 0.16)}",
                    opacity=1.0,
                    boxpoints='all',
                    jitter=0.28,
                    pointpos=0.0,
                    hovertemplate=f'{interval}<br>{grp}<br>{TARGET_ASSEMBLY}: %{{y:.4f}}<extra></extra>',
                ),
                row=1,
                col=col_idx,
            )

        fig.update_xaxes(
            categoryorder='array',
            categoryarray=list(group_order),
            row=1,
            col=col_idx,
        )

        xdom = 'x domain' if col_idx == 1 else f'x{col_idx} domain'
        ydom = 'y domain' if col_idx == 1 else f'y{col_idx} domain'

        if interval in summary_map.index:
            row = summary_map.loc[interval]
            ann = '<br>'.join([
                f"n={int(row['n_a'])}/{int(row['n_b'])}",
                f"t={row['welch_t']:.3g}",
                f"p (2-sided)={row['welch_p_two_sided']:.3g}",
            ])
            fig.add_annotation(
                x=0.02, y=0.98,
                xref=xdom, yref=ydom,
                xanchor='left', yanchor='top',
                text=ann,
                showarrow=False,
                align='left',
                font=dict(size=10),
                bgcolor='rgba(255,255,255,0.8)',
                bordercolor='rgba(0,0,0,0.08)',
            )

    fig.update_yaxes(title_text=f'Mean {TARGET_ASSEMBLY} activation per trial/interval', row=1, col=1)
    fig.update_layout(
        title=title,
        template='plotly_white',
        height=470,
        width=max(1000, 340 * n_cols),
        margin=dict(l=80, r=30, t=80, b=70),
    )

    if show:
        display(fig)
        return None
    return fig


plot_overall_group_comparison(
    trial_means_ens12,
    overall_cue1_vs_cue2,
    group_col='cue_label',
    group_order=['cue1', 'cue2'],
    title=f'{TARGET_ASSEMBLY}: expert trials, cue1 vs cue2 (pooled across sessions)',
    colors=['#f28e2b', '#7b2cbf'],
)






Overall comparison (expert trials): cue1 vs cue2 (group_b - group_a = cue2 - cue1)


,interval_name,group_a,group_b,n_a,n_b,mean_a,mean_b,std_a,std_b,diff_b_minus_a,welch_t,welch_p_two_sided,welch_p_one_sided_b_gt_a,mw_u,mw_p_two_sided
0,R1_entry_interval,cue1,cue2,371,430,1.270207,1.244805,0.885600,0.829060,-0.025402,0.416910,0.676861,0.338431,79844.0,0.980820
1,R2_entry_interval,cue1,cue2,371,430,1.631879,1.359677,1.098429,0.925040,-0.272202,3.759557,0.000184,0.000092,91289.0,0.000417
2,cue_entry_interval,cue1,cue2,371,430,1.370969,1.092500,1.017842,0.842448,-0.278469,4.177736,0.000033,0.000017,91692.0,0.000260


In [7]:
# Session-wise cue-only comparisons and time trends (expert trials)

session_cue1_vs_cue2 = summarize_by_session(trial_means_ens12, 'cue_label', 'cue1', 'cue2')
trend_cue1_vs_cue2 = summarize_trend(session_cue1_vs_cue2)

print('Session-wise trend summary (expert trials): cue1 vs cue2 (diff = cue2 - cue1)')
display(trend_cue1_vs_cue2)


def plot_session_differences(session_df, title, y_label, diff_col='diff_b_minus_a', show=True):
    n_rows = len(TARGET_INTERVALS)
    fig = make_subplots(
        rows=n_rows,
        cols=1,
        shared_xaxes=True,
        subplot_titles=TARGET_INTERVALS,
        vertical_spacing=0.07,
    )

    for row_idx, interval in enumerate(TARGET_INTERVALS, start=1):
        sub = session_df.loc[
            (session_df['interval_name'] == interval) & session_df[diff_col].notna()
        ].sort_values('session_order')

        fig.add_hline(y=0, line_color='rgba(0,0,0,0.4)', line_width=1, row=row_idx, col=1)

        x = sub['session_order'].to_numpy()
        y = sub[diff_col].to_numpy()
        pvals = sub['welch_p_two_sided'].to_numpy()
        sig = np.isfinite(pvals) & (pvals < 0.05)

        fig.add_trace(
            go.Scatter(
                x=x, y=y,
                mode='lines',
                line=dict(color='#3a86ff', width=1.6),
                name='Difference',
                showlegend=(row_idx == 1),
                hovertemplate='Session order: %{x}<br>Diff: %{y:.4f}<extra></extra>',
            ),
            row=row_idx, col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=x[~sig], y=y[~sig],
                mode='markers',
                marker=dict(color='#3a86ff', size=6, opacity=0.65),
                name='p >= 0.05',
                showlegend=(row_idx == 1),
                hovertemplate='Session order: %{x}<br>Diff: %{y:.4f}<extra></extra>',
            ),
            row=row_idx, col=1,
        )
        if np.any(sig):
            fig.add_trace(
                go.Scatter(
                    x=x[sig], y=y[sig],
                    mode='markers',
                    marker=dict(color='#d62828', size=7, opacity=0.95),
                    name='p < 0.05',
                    showlegend=(row_idx == 1),
                    hovertemplate='Session order: %{x}<br>Diff: %{y:.4f}<extra></extra>',
                ),
                row=row_idx, col=1,
            )

        if len(sub) >= 3 and np.isfinite(y).sum() >= 3:
            try:
                rho, p_val = stats.spearmanr(sub['session_order'], sub[diff_col])
                trend_txt = f"Spearman rho={rho:.2f}, p={p_val:.3g}" if pd.notna(rho) else 'Spearman: n/a'
            except Exception:
                trend_txt = 'Spearman: n/a'
        else:
            trend_txt = 'Spearman: insufficient sessions'

        xdom = 'x domain' if row_idx == 1 else f'x{row_idx} domain'
        ydom = 'y domain' if row_idx == 1 else f'y{row_idx} domain'
        fig.add_annotation(
            x=0.01, y=0.95,
            xref=xdom, yref=ydom,
            xanchor='left', yanchor='top',
            text=trend_txt,
            showarrow=False,
            font=dict(size=10),
            bgcolor='rgba(255,255,255,0.75)',
        )
        fig.update_yaxes(title_text=y_label, row=row_idx, col=1)

    all_sessions = (
        session_df[['session_id', 'session_order']]
        .drop_duplicates()
        .sort_values('session_order')
    )
    fig.update_xaxes(
        tickmode='array',
        tickvals=all_sessions['session_order'].tolist(),
        ticktext=all_sessions['session_id'].tolist(),
        tickangle=90,
        row=n_rows,
        col=1,
    )
    fig.update_xaxes(title_text='Session (chronological order)', row=n_rows, col=1)

    fig.update_layout(
        title=title,
        template='plotly_white',
        height=max(720, 240 * n_rows),
        width=1180,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0),
        margin=dict(l=80, r=30, t=85, b=120),
    )

    if show:
        display(fig)
        return None
    return fig


plot_session_differences(
    session_cue1_vs_cue2,
    title=f'{TARGET_ASSEMBLY}: session-wise difference over time (cue2 - cue1, expert trials)',
    y_label='Mean diff (group_b - group_a)',
)




Session-wise trend summary (expert trials): cue1 vs cue2 (diff = cue2 - cue1)


,interval_name,n_sessions,spearman_rho,spearman_p
0,R1_entry_interval,26,-0.159658,0.435933
1,R2_entry_interval,26,-0.167179,0.414331
2,cue_entry_interval,26,0.044786,0.828018


___
___
# Trials based tests for significance

### Questions asked: 
- Is enesmble more active in C1 or C2 trials? 


In [8]:
# Trial-number based statistics (trial_id within session, pooled across sessions)
def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    q = np.full(p.shape, np.nan, dtype=float)
    mask = np.isfinite(p)
    if mask.sum() == 0:
        return q

    p_valid = p[mask]
    order = np.argsort(p_valid)
    ranked_p = p_valid[order]
    m = len(ranked_p)

    ranked_q = ranked_p * m / np.arange(1, m + 1)
    ranked_q = np.minimum.accumulate(ranked_q[::-1])[::-1]
    ranked_q = np.clip(ranked_q, 0, 1)

    q_valid = np.empty_like(p_valid)
    q_valid[order] = ranked_q
    q[mask] = q_valid
    return q


def add_fdr_by_groups(df, group_cols, p_col='welch_p_two_sided', out_col='welch_fdr_q'):
    out = df.copy()
    out[out_col] = np.nan
    for _, idx in out.groupby(group_cols, dropna=False).groups.items():
        idx = list(idx)
        out.loc[idx, out_col] = benjamini_hochberg(out.loc[idx, p_col].to_numpy())
    return out


def summarize_by_trial_index(trial_means, group_col, group_a, group_b, intervals=TARGET_INTERVALS):
    rows = []
    grouped = trial_means.groupby(['trial_id', 'interval_name'], sort=True, dropna=False)
    for (trial_id, interval), sub in grouped:
        if interval not in intervals:
            continue
        stats_row = compare_two_groups(sub, group_col, group_a, group_b)
        rows.append({
            'trial_id': int(trial_id) if pd.notna(trial_id) else trial_id,
            'interval_name': interval,
            'group_a': group_a,
            'group_b': group_b,
            'n_sessions_a': sub.loc[sub[group_col] == group_a, 'session_id'].nunique(),
            'n_sessions_b': sub.loc[sub[group_col] == group_b, 'session_id'].nunique(),
            **stats_row,
        })

    out = pd.DataFrame(rows)
    out = out.sort_values(['interval_name', 'trial_id']).reset_index(drop=True)
    out = add_fdr_by_groups(out, ['interval_name'], p_col='welch_p_two_sided', out_col='welch_fdr_q')
    out = add_fdr_by_groups(out, ['interval_name'], p_col='mw_p_two_sided', out_col='mw_fdr_q')
    return out


def summarize_trial_index_trend(summary_df, diff_col='diff_b_minus_a'):
    rows = []
    for interval in TARGET_INTERVALS:
        sub = summary_df.loc[
            (summary_df['interval_name'] == interval) &
            summary_df[diff_col].notna()
        ].sort_values('trial_id')
        if len(sub) >= 3:
            rho, p_val = stats.spearmanr(sub['trial_id'], sub[diff_col])
        else:
            rho, p_val = (np.nan, np.nan)
        rows.append({
            'interval_name': interval,
            'n_trial_ids': len(sub),
            'spearman_rho': float(rho) if pd.notna(rho) else np.nan,
            'spearman_p': float(p_val) if pd.notna(p_val) else np.nan,
        })
    return pd.DataFrame(rows)


def summarize_trial_index_group_trajectory(trial_means, group_col, group_order, value_col='assembly_mean'):
    g = (
        trial_means.groupby(['interval_name', 'trial_id', group_col], dropna=False)[value_col]
        .agg(['mean', 'std', 'count'])
        .reset_index()
        .rename(columns={'count': 'n'})
    )
    g['sem'] = g['std'] / np.sqrt(g['n'].where(g['n'] > 0, np.nan))
    g[group_col] = pd.Categorical(g[group_col], categories=group_order, ordered=True)
    return g.sort_values(['interval_name', 'trial_id', group_col]).reset_index(drop=True)


def plot_trial_index_differences(summary_df, title, diff_col='diff_b_minus_a', q_col='welch_fdr_q', show=True):
    n_rows = len(TARGET_INTERVALS)
    fig = make_subplots(
        rows=n_rows,
        cols=1,
        shared_xaxes=True,
        subplot_titles=TARGET_INTERVALS,
        vertical_spacing=0.07,
    )

    for row_idx, interval in enumerate(TARGET_INTERVALS, start=1):
        sub = summary_df.loc[
            (summary_df['interval_name'] == interval) & summary_df[diff_col].notna()
        ].sort_values('trial_id')
        fig.add_hline(y=0, line_color='rgba(0,0,0,0.4)', line_width=1, row=row_idx, col=1)

        x = sub['trial_id'].to_numpy()
        y = sub[diff_col].to_numpy()
        y_roll = pd.Series(y).rolling(window=5, min_periods=2, center=True).mean().to_numpy()
        q = sub[q_col].to_numpy()
        sig = np.isfinite(q) & (q < 0.05)

        fig.add_trace(
            go.Scatter(
                x=x, y=y, mode='lines',
                line=dict(color='#1d3557', width=1.2),
                name='Raw difference',
                showlegend=(row_idx == 1),
                hovertemplate='trial_id=%{x}<br>Diff=%{y:.4f}<extra></extra>',
            ),
            row=row_idx, col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=x, y=y_roll, mode='lines',
                line=dict(color='#457b9d', width=2.0),
                name='Rolling mean (5)',
                showlegend=(row_idx == 1),
                hovertemplate='trial_id=%{x}<br>Rolling diff=%{y:.4f}<extra></extra>',
            ),
            row=row_idx, col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=x[~sig], y=y[~sig], mode='markers',
                marker=dict(size=6, color='#457b9d', opacity=0.45),
                name='q >= 0.05',
                showlegend=(row_idx == 1),
                hovertemplate='trial_id=%{x}<br>Diff=%{y:.4f}<extra></extra>',
            ),
            row=row_idx, col=1,
        )
        if np.any(sig):
            fig.add_trace(
                go.Scatter(
                    x=x[sig], y=y[sig], mode='markers',
                    marker=dict(size=7, color='#d62828', opacity=0.95),
                    name='q < 0.05',
                    showlegend=(row_idx == 1),
                    hovertemplate='trial_id=%{x}<br>Diff=%{y:.4f}<extra></extra>',
                ),
                row=row_idx, col=1,
            )

        txt = f"trial_ids={len(sub)}, sig(q<0.05)={int(sig.sum())}"
        xdom = 'x domain' if row_idx == 1 else f'x{row_idx} domain'
        ydom = 'y domain' if row_idx == 1 else f'y{row_idx} domain'
        fig.add_annotation(
            x=0.01, y=0.95,
            xref=xdom, yref=ydom,
            xanchor='left', yanchor='top',
            text=txt,
            showarrow=False,
            font=dict(size=10),
            bgcolor='rgba(255,255,255,0.75)',
        )
        fig.update_yaxes(title_text='Mean diff<br>(group_b - group_a)', row=row_idx, col=1)

    fig.update_xaxes(title_text='trial_id (within-session trial number)', row=n_rows, col=1)
    fig.update_layout(
        title=title,
        template='plotly_white',
        height=max(760, 250 * n_rows),
        width=1180,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0),
        margin=dict(l=90, r=30, t=90, b=80),
    )

    if show:
        display(fig)
        return None
    return fig


def plot_trial_index_group_trajectories(traj_df, summary_df, group_col, group_order, title, colors=None, q_col='welch_fdr_q', show=True):
    if colors is None:
        colors = ['#2a9d8f', '#e76f51']
    color_map = dict(zip(group_order, colors))

    n_rows = len(TARGET_INTERVALS)
    fig = make_subplots(
        rows=n_rows,
        cols=1,
        shared_xaxes=True,
        subplot_titles=TARGET_INTERVALS,
        vertical_spacing=0.07,
    )

    for row_idx, interval in enumerate(TARGET_INTERVALS, start=1):
        sub_traj = traj_df.loc[traj_df['interval_name'] == interval].copy()
        sub_sig = summary_df.loc[summary_df['interval_name'] == interval].sort_values('trial_id').copy()
        ymins = []
        ymaxs = []

        for grp in group_order:
            g = sub_traj.loc[sub_traj[group_col] == grp].sort_values('trial_id')
            x = g['trial_id'].to_numpy()
            mean = g['mean'].to_numpy(dtype=float)
            sem = g['sem'].fillna(0).to_numpy(dtype=float)
            ymins.append(np.nanmin(mean - sem))
            ymaxs.append(np.nanmax(mean + sem))
            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=mean,
                    mode='lines+markers',
                    line=dict(color=color_map[grp], width=2.0),
                    marker=dict(color=color_map[grp], size=5, opacity=0.6),
                    error_y=dict(type='data', array=sem, visible=True, thickness=0.6, width=0),
                    name=str(grp),
                    legendgroup=str(grp),
                    showlegend=(row_idx == 1),
                    hovertemplate=f'{interval}<br>{grp}<br>trial_id=%{{x}}<br>Mean=%{{y:.4f}}<extra></extra>',
                ),
                row=row_idx,
                col=1,
            )

        if ymins and ymaxs:
            q = sub_sig[q_col].to_numpy()
            sig = np.isfinite(q) & (q < 0.05)
            if np.any(sig):
                ymin = float(np.nanmin(ymins))
                ymax = float(np.nanmax(ymaxs))
                yspan = ymax - ymin if np.isfinite(ymax - ymin) and ymax > ymin else 1.0
                y_mark = ymin - 0.05 * yspan
                fig.add_trace(
                    go.Scatter(
                        x=sub_sig.loc[sig, 'trial_id'],
                        y=np.full(int(sig.sum()), y_mark),
                        mode='markers',
                        marker=dict(size=6, color='#d62828', symbol='diamond'),
                        name='q < 0.05',
                        showlegend=(row_idx == 1),
                        hovertemplate='trial_id=%{x}<br>Significant (q<0.05)<extra></extra>',
                    ),
                    row=row_idx,
                    col=1,
                )

        fig.update_yaxes(title_text=f'Mean {TARGET_ASSEMBLY}', row=row_idx, col=1)

    fig.update_xaxes(title_text='trial_id (within-session trial number)', row=n_rows, col=1)
    fig.update_layout(
        title=title,
        template='plotly_white',
        height=max(760, 260 * n_rows),
        width=1180,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0),
        margin=dict(l=90, r=30, t=90, b=80),
    )

    if show:
        display(fig)
        return None
    return fig





In [9]:
# Trial-number based cue comparison (expert trials): significance over trial progression

trial_index_cue1_vs_cue2 = summarize_by_trial_index(trial_means_ens12, 'cue_label', 'cue1', 'cue2')
trial_index_trend_cue1_vs_cue2 = summarize_trial_index_trend(trial_index_cue1_vs_cue2)

print('Trial-index trend summary (expert trials): cue1 vs cue2 (diff = cue2 - cue1)')
display(trial_index_trend_cue1_vs_cue2)

print()
print('Significant trial_ids (Welch FDR q < 0.05), cue1 vs cue2:')
display(trial_index_cue1_vs_cue2.loc[trial_index_cue1_vs_cue2['welch_fdr_q'] < 0.05].head(20))
print('... total significant rows:', int((trial_index_cue1_vs_cue2['welch_fdr_q'] < 0.05).sum()))

plot_trial_index_differences(
    trial_index_cue1_vs_cue2,
    title=f'{TARGET_ASSEMBLY}: trial-number based difference (cue2 - cue1, expert trials, pooled across sessions)',
)



Trial-index trend summary (expert trials): cue1 vs cue2 (diff = cue2 - cue1)


,interval_name,n_trial_ids,spearman_rho,spearman_p
0,R1_entry_interval,113,0.014871,0.875769
1,R2_entry_interval,113,0.133583,0.158377
2,cue_entry_interval,113,0.226412,0.015890



Significant trial_ids (Welch FDR q < 0.05), cue1 vs cue2:


,trial_id,interval_name,group_a,group_b,n_sessions_a,n_sessions_b,n_a,n_b,mean_a,mean_b,std_a,std_b,diff_b_minus_a,welch_t,welch_p_two_sided,welch_p_one_sided_b_gt_a,mw_u,mw_p_two_sided,welch_fdr_q,mw_fdr_q


... total significant rows: 0


In [10]:
# --- Trial-number based raw trajectories (expert trials, cue-only) ---

trial_traj_cue1_vs_cue2 = summarize_trial_index_group_trajectory(
    trial_means_ens12,
    group_col='cue_label',
    group_order=['cue1', 'cue2'],
)

plot_trial_index_group_trajectories(
    trial_traj_cue1_vs_cue2,
    trial_index_cue1_vs_cue2,
    group_col='cue_label',
    group_order=['cue1', 'cue2'],
    title=f'{TARGET_ASSEMBLY}: trial-number mean trajectories (cue1 vs cue2, expert trials)',
    colors=['#f28e2b', '#7b2cbf'],
)



# Withing trial tests for significance

## Questions asked:
- Within trials along progression of the intervals, can we see emerging differences?

In [11]:
# Within-interval binwise data prep + interval length checks
def prepare_expert_binwise_interval_data(t0_ens, assembly_col=TARGET_ASSEMBLY, intervals=TARGET_INTERVALS):
    df = _analytics_df_with_columns(t0_ens)

    needed = [
        'session_id', 'trial_id', 'cue', 'choice_R1', 'choice_R2', 'interval_name',
        'from_ephys_timestamp', 'to_ephys_timestamp', assembly_col
    ]
    optional = [c for c in ['entry_id', 't0_event_name'] if c in df.columns]

    df = df[needed + optional].copy()
    df = df[df['interval_name'].isin(intervals)].copy()
    df['cue'] = pd.to_numeric(df['cue'], errors='coerce')
    df = df[df[assembly_col].notna()].copy()

    # only expert trials
    expert_mask = (
        ((df['cue'] == 1) & df['choice_R1'].eq(True) & df['choice_R2'].eq(False)) |
        ((df['cue'] == 2) & df['choice_R1'].eq(False) & df['choice_R2'].eq(True))
    )
    df = df.loc[expert_mask].copy()

    df['cue_label'] = df['cue'].map({1.0: 'cue1', 2.0: 'cue2'})
    df['r1_action'] = np.where(df['choice_R1'].eq(True), 'stop_R1', 'skip_R1')
    df['assembly_value'] = df[assembly_col].astype(float)
    df['bin_width_us'] = (df['to_ephys_timestamp'] - df['from_ephys_timestamp']).astype(float)

    sort_cols = ['session_id', 'trial_id', 'interval_name', 'from_ephys_timestamp', 'to_ephys_timestamp']
    if 'entry_id' in df.columns:
        sort_cols.append('entry_id')
    df = df.sort_values(sort_cols).reset_index(drop=True)

    group_cols = ['session_id', 'trial_id', 'interval_name']
    df['bin_idx'] = df.groupby(group_cols).cumcount().astype(int)
    df['n_bins'] = df.groupby(group_cols)['assembly_value'].transform('size').astype(int)
    df['first_from_us'] = df.groupby(group_cols)['from_ephys_timestamp'].transform('min').astype(float)
    df['last_to_us'] = df.groupby(group_cols)['to_ephys_timestamp'].transform('max').astype(float)
    df['interval_duration_us'] = df['last_to_us'] - df['first_from_us']
    df['bin_center_us'] = (df['from_ephys_timestamp'].astype(float) + df['to_ephys_timestamp'].astype(float)) / 2.0
    df['bin_center_ms'] = (df['bin_center_us'] - df['first_from_us']) / 1000.0
    df['bin_center_frac'] = (df['bin_idx'] + 0.5) / df['n_bins'].replace(0, np.nan)

    interval_trial_meta = (
        df.groupby(group_cols, dropna=False)
        .agg(
            cue=('cue', 'first'),
            cue_label=('cue_label', 'first'),
            r1_action=('r1_action', 'first'),
            choice_R1=('choice_R1', 'first'),
            choice_R2=('choice_R2', 'first'),
            n_bins=('n_bins', 'first'),
            first_from_us=('from_ephys_timestamp', 'min'),
            last_to_us=('to_ephys_timestamp', 'max'),
            interval_duration_us=('interval_duration_us', 'first'),
            bin_width_us_mode=('bin_width_us', lambda s: s.mode().iloc[0]),
        )
        .reset_index()
    )
    interval_trial_meta['session_dt'] = pd.to_datetime(interval_trial_meta['session_id'], format='%Y-%m-%d_%H-%M', errors='coerce')

    return df, interval_trial_meta


def summarize_interval_length_checks(interval_trial_meta):
    per_session = (
        interval_trial_meta.groupby(['session_id', 'interval_name'], dropna=False)
        .agg(
            n_trials=('trial_id', 'nunique'),
            unique_n_bins=('n_bins', 'nunique'),
            min_n_bins=('n_bins', 'min'),
            max_n_bins=('n_bins', 'max'),
            unique_duration_us=('interval_duration_us', 'nunique'),
            min_duration_us=('interval_duration_us', 'min'),
            max_duration_us=('interval_duration_us', 'max'),
            median_duration_us=('interval_duration_us', 'median'),
        )
        .reset_index()
        .sort_values(['session_id', 'interval_name'])
        .reset_index(drop=True)
    )
    per_session['n_bins_consistent'] = per_session['unique_n_bins'].eq(1)
    per_session['duration_consistent'] = per_session['unique_duration_us'].eq(1)

    overall = (
        interval_trial_meta.groupby(['interval_name'], dropna=False)
        .agg(
            n_interval_trials=('trial_id', 'size'),
            n_sessions=('session_id', 'nunique'),
            unique_n_bins=('n_bins', 'nunique'),
            min_n_bins=('n_bins', 'min'),
            max_n_bins=('n_bins', 'max'),
            unique_duration_us=('interval_duration_us', 'nunique'),
            min_duration_us=('interval_duration_us', 'min'),
            max_duration_us=('interval_duration_us', 'max'),
            median_duration_us=('interval_duration_us', 'median'),
        )
        .reset_index()
        .sort_values('interval_name')
        .reset_index(drop=True)
    )
    overall['n_bins_consistent'] = overall['unique_n_bins'].eq(1)
    overall['duration_consistent'] = overall['unique_duration_us'].eq(1)
    return per_session, overall


def plot_interval_duration_checks(interval_trial_meta, show=True):
    n_rows = len(TARGET_INTERVALS)
    fig = make_subplots(
        rows=n_rows,
        cols=1,
        shared_xaxes=True,
        subplot_titles=TARGET_INTERVALS,
        vertical_spacing=0.07,
    )

    for row_idx, interval in enumerate(TARGET_INTERVALS, start=1):
        sub = interval_trial_meta.loc[interval_trial_meta['interval_name'] == interval].copy()
        for session_id, g in sub.groupby('session_id', sort=False):
            g = g.sort_values('trial_id')
            fig.add_trace(
                go.Scatter(
                    x=g['trial_id'],
                    y=g['interval_duration_us'] / 1000.0,
                    mode='lines',
                    line=dict(color='rgba(141,153,174,0.22)', width=1),
                    showlegend=False,
                    hovertemplate=f'{interval}<br>session={session_id}<br>trial_id=%{{x}}<br>Duration=%{{y:.1f}} ms<extra></extra>',
                ),
                row=row_idx, col=1,
            )

        med = sub.groupby('trial_id', dropna=False)['interval_duration_us'].median().reset_index()
        fig.add_trace(
            go.Scatter(
                x=med['trial_id'],
                y=med['interval_duration_us'] / 1000.0,
                mode='lines',
                line=dict(color='#1d3557', width=2.2),
                name='Median across sessions',
                showlegend=(row_idx == 1),
                hovertemplate=f'{interval}<br>trial_id=%{{x}}<br>Median duration=%{{y:.1f}} ms<extra></extra>',
            ),
            row=row_idx, col=1,
        )
        fig.update_yaxes(title_text='Duration (ms)', row=row_idx, col=1)

    fig.update_xaxes(title_text='trial_id (within-session trial number)', row=n_rows, col=1)
    fig.update_layout(
        title='Interval duration check: last to_ephys_timestamp - first from_ephys_timestamp',
        template='plotly_white',
        height=max(720, 230 * n_rows),
        width=1180,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0),
        margin=dict(l=80, r=30, t=90, b=80),
    )

    if show:
        display(fig)
        return None
    return fig


expert_bins_ens12, interval_trial_meta_ens12 = prepare_expert_binwise_interval_data(t0_ens)
length_check_session_ens12, length_check_overall_ens12 = summarize_interval_length_checks(interval_trial_meta_ens12)

print('Interval length check (overall):')
display(length_check_overall_ens12)

print()
print('Session/interval rows with inconsistent bin count or duration (first 30 shown):')
incons = length_check_session_ens12.loc[
    ~length_check_session_ens12['n_bins_consistent'] | ~length_check_session_ens12['duration_consistent']
]
display(incons.head(30))
print('... total inconsistent session/interval rows:', len(incons))

plot_interval_duration_checks(interval_trial_meta_ens12)




Interval length check (overall):


,interval_name,n_interval_trials,n_sessions,unique_n_bins,min_n_bins,max_n_bins,unique_duration_us,min_duration_us,max_duration_us,median_duration_us,n_bins_consistent,duration_consistent
0,R1_entry_interval,801,28,1,45,45,1,1800000.0,1800000.0,1800000.0,True,True
1,R2_entry_interval,801,28,1,40,40,1,1600000.0,1600000.0,1600000.0,True,True
2,cue_entry_interval,801,28,1,40,40,1,1600000.0,1600000.0,1600000.0,True,True



Session/interval rows with inconsistent bin count or duration (first 30 shown):


,session_id,interval_name,n_trials,unique_n_bins,min_n_bins,max_n_bins,unique_duration_us,min_duration_us,max_duration_us,median_duration_us,n_bins_consistent,duration_consistent


... total inconsistent session/interval rows: 0


In [12]:
# Within-interval progression tests: bin-aligned significance across expert trials (cue-only)
def summarize_binwise_overall(expert_bins_df, group_col, group_a, group_b, intervals=TARGET_INTERVALS, window_half_bins=1):
    rows = []
    for interval in intervals:
        interval_df = expert_bins_df.loc[expert_bins_df['interval_name'] == interval].copy()
        bin_ids = sorted(pd.Series(interval_df['bin_idx'].dropna().unique()).astype(int).tolist())
        for bin_idx in bin_ids:
            center = interval_df.loc[interval_df['bin_idx'] == bin_idx].copy()
            window = interval_df.loc[
                interval_df['bin_idx'].between(bin_idx - window_half_bins, bin_idx + window_half_bins)
            ].copy()
            stats_row = compare_two_groups(window, group_col, group_a, group_b, value_col='assembly_value')
            rows.append({
                'interval_name': interval,
                'bin_idx': int(bin_idx),
                'group_a': group_a,
                'group_b': group_b,
                'bin_center_ms': float(center['bin_center_ms'].median()) if center['bin_center_ms'].notna().any() else np.nan,
                'n_bins_mode': float(center['n_bins'].mode().iloc[0]),
                'window_half_bins': int(window_half_bins),
                'window_n_bins_used': int(window['bin_idx'].nunique()),
                'n_sessions_a': window.loc[window[group_col] == group_a, 'session_id'].nunique(),
                'n_sessions_b': window.loc[window[group_col] == group_b, 'session_id'].nunique(),
                **stats_row,
            })
    out = pd.DataFrame(rows)
    out = out.sort_values(['interval_name', 'bin_idx']).reset_index(drop=True)
    out = add_fdr_by_groups(out, ['interval_name'], p_col='welch_p_two_sided', out_col='welch_fdr_q')
    out = add_fdr_by_groups(out, ['interval_name'], p_col='mw_p_two_sided', out_col='mw_fdr_q')
    return out


def summarize_binwise_group_trajectory(expert_bins_df, group_col, group_order):
    traj = (
        expert_bins_df.groupby(['interval_name', 'bin_idx', group_col], dropna=False)['assembly_value']
        .agg(['mean', 'std', 'count'])
        .reset_index()
        .rename(columns={'count': 'n'})
    )
    centers = (
        expert_bins_df.groupby(['interval_name', 'bin_idx'], dropna=False)['bin_center_ms']
        .median()
        .reset_index()
    )
    traj = traj.merge(centers, on=['interval_name', 'bin_idx'], how='left')
    traj['sem'] = traj['std'] / np.sqrt(traj['n'].where(traj['n'] > 0, np.nan))
    traj[group_col] = pd.Categorical(traj[group_col], categories=group_order, ordered=True)
    return traj.sort_values(['interval_name', 'bin_idx', group_col]).reset_index(drop=True)


def plot_binwise_trajectories_with_significance(traj_df, summary_panels, group_col, group_order, title, colors=None, q_col='welch_fdr_q', show=True):
    if colors is None:
        colors = ['#f28e2b', '#7b2cbf']
    color_map = dict(zip(group_order, colors))

    if isinstance(summary_panels, pd.DataFrame):
        panels = [
            {
                'summary_df': summary_panels,
                'panel_title': 'Binwise test',
                'sig_label': 'q < 0.05',
                'window_desc': '',
            }
        ]
    else:
        panels = []
        for i, spec in enumerate(summary_panels, start=1):
            if isinstance(spec, pd.DataFrame):
                panels.append({
                    'summary_df': spec,
                    'panel_title': f'Panel {i}',
                    'sig_label': 'q < 0.05',
                    'window_desc': '',
                })
            else:
                panels.append({
                    'summary_df': spec['summary_df'],
                    'panel_title': spec.get('panel_title', f'Panel {i}'),
                    'sig_label': spec.get('sig_label', 'q < 0.05'),
                    'window_desc': spec.get('window_desc', ''),
                })

    n_rows = len(TARGET_INTERVALS)
    n_cols = len(panels)
    subplot_titles = [
        f'{interval} | {panel["panel_title"]}'
        for interval in TARGET_INTERVALS
        for panel in panels
    ]
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        shared_xaxes=False,
        shared_yaxes=False,
        subplot_titles=subplot_titles,
        vertical_spacing=0.07,
        horizontal_spacing=0.08,
    )

    for row_idx, interval in enumerate(TARGET_INTERVALS, start=1):
        sub_traj = traj_df.loc[traj_df['interval_name'] == interval].copy()
        group_plot_data = {}
        ymins, ymaxs = [], []

        for grp in group_order:
            g = sub_traj.loc[sub_traj[group_col] == grp].sort_values('bin_idx')
            x = g['bin_center_ms'].to_numpy(dtype=float)
            mean = g['mean'].to_numpy(dtype=float)
            sem = g['sem'].fillna(0).to_numpy(dtype=float)
            group_plot_data[grp] = (x, mean, sem)
            finite = np.isfinite(mean) & np.isfinite(sem)
            if np.any(finite):
                ymins.append(float(np.nanmin(mean[finite] - sem[finite])))
                ymaxs.append(float(np.nanmax(mean[finite] + sem[finite])))

        if ymins and ymaxs:
            ymin = float(np.nanmin(ymins))
            ymax = float(np.nanmax(ymaxs))
        else:
            ymin, ymax = -0.5, 0.5
        yspan = ymax - ymin if np.isfinite(ymax - ymin) and ymax > ymin else 1.0
        y_mark = ymin - 0.05 * yspan
        y_range = [y_mark - 0.08 * yspan, ymax + 0.05 * yspan]

        for col_idx, panel in enumerate(panels, start=1):
            sub_sig = panel['summary_df'].loc[panel['summary_df']['interval_name'] == interval].sort_values('bin_idx').copy()

            for grp in group_order:
                x, mean, sem = group_plot_data[grp]
                fig.add_trace(
                    go.Scatter(
                        x=x,
                        y=mean,
                        mode='lines+markers',
                        line=dict(color=color_map[grp], width=2.0),
                        marker=dict(color=color_map[grp], size=5, opacity=0.7),
                        error_y=dict(type='data', array=sem, visible=True, thickness=0.6, width=0),
                        name=str(grp),
                        legendgroup=str(grp),
                        showlegend=(row_idx == 1 and col_idx == 1),
                        hovertemplate=f'{interval}<br>{grp}<br>Time=%{{x:.1f}} ms<br>Mean=%{{y:.4f}}<extra></extra>',
                    ),
                    row=row_idx, col=col_idx,
                )

            q = sub_sig[q_col].to_numpy(dtype=float) if q_col in sub_sig.columns else np.full(len(sub_sig), np.nan)
            sig = np.isfinite(q) & (q < 0.05)
            if np.any(sig):
                window_desc = panel['window_desc'] if panel['window_desc'] else panel['panel_title']
                fig.add_trace(
                    go.Scatter(
                        x=sub_sig.loc[sig, 'bin_center_ms'],
                        y=np.full(int(sig.sum()), y_mark),
                        mode='markers',
                        marker=dict(size=6, color='#d62828', symbol='diamond'),
                        name=panel['sig_label'],
                        legendgroup=f"sig::{panel['sig_label']}",
                        showlegend=(row_idx == 1),
                        hovertemplate=f"Time=%{{x:.1f}} ms<br>Significant ({window_desc}; Welch FDR q<0.05)<extra></extra>",
                    ),
                    row=row_idx, col=col_idx,
                )

            txt = f"bins={len(sub_sig)}, sig(q<0.05)={int(sig.sum())}"
            if panel['window_desc']:
                txt = f"{txt}, {panel['window_desc']}"
            axis_num = (row_idx - 1) * n_cols + col_idx
            xdom = 'x domain' if axis_num == 1 else f'x{axis_num} domain'
            ydom = 'y domain' if axis_num == 1 else f'y{axis_num} domain'
            fig.add_annotation(
                x=0.01, y=0.95,
                xref=xdom, yref=ydom,
                xanchor='left', yanchor='top',
                text=txt,
                showarrow=False,
                font=dict(size=10),
                bgcolor='rgba(255,255,255,0.75)',
            )
            fig.update_yaxes(range=y_range, row=row_idx, col=col_idx)
            if col_idx == 1:
                fig.update_yaxes(title_text=f'{TARGET_ASSEMBLY}', row=row_idx, col=col_idx)

    for col_idx in range(1, n_cols + 1):
        fig.update_xaxes(title_text='Time from interval start (ms)', row=n_rows, col=col_idx)

    fig.update_layout(
        title=title,
        template='plotly_white',
        height=max(780, 260 * n_rows),
        width=max(1180, 700 * n_cols),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0),
        margin=dict(l=80, r=30, t=100, b=80),
    )

    if show:
        display(fig)
        return None
    return fig


binwise_cue1_vs_cue2_raw = summarize_binwise_overall(expert_bins_ens12, 'cue_label', 'cue1', 'cue2', window_half_bins=0)
binwise_cue1_vs_cue2 = summarize_binwise_overall(expert_bins_ens12, 'cue_label', 'cue1', 'cue2', window_half_bins=1)
bintraj_cue1_vs_cue2 = summarize_binwise_group_trajectory(expert_bins_ens12, 'cue_label', ['cue1', 'cue2'])

print('Within-interval binwise significance summary (expert trials, cue1 vs cue2; Welch tests use center±1 bin window):')
display(
    binwise_cue1_vs_cue2.groupby('interval_name', dropna=False)
    .agg(
        n_bins_tested=('bin_idx', 'size'),
        n_sig_welch_fdr=('welch_fdr_q', lambda s: int((s < 0.05).sum())),
        median_abs_diff=('diff_b_minus_a', lambda s: float(np.nanmedian(np.abs(s)))),
    )
    .reset_index()
)

plot_binwise_trajectories_with_significance(
    bintraj_cue1_vs_cue2,
    [
        {
            'summary_df': binwise_cue1_vs_cue2_raw,
            'panel_title': 'Raw bins',
            'sig_label': 'q < 0.05 (raw bin)',
            'window_desc': 'center bin only',
        },
        {
            'summary_df': binwise_cue1_vs_cue2,
            'panel_title': '3-bin window',
            'sig_label': 'q < 0.05 (3-bin window)',
            'window_desc': 'center±1 bin',
        },
    ],
    group_col='cue_label',
    group_order=['cue1', 'cue2'],
    title=f'{TARGET_ASSEMBLY}: within-interval progression (cue1 vs cue2, expert trials) | raw-bin vs center±1-bin significance',
    colors=['#f28e2b', '#7b2cbf'],
)






Within-interval binwise significance summary (expert trials, cue1 vs cue2; Welch tests use center±1 bin window):


,interval_name,n_bins_tested,n_sig_welch_fdr,median_abs_diff
0,R1_entry_interval,45,2,0.125450
1,R2_entry_interval,40,24,0.332457
2,cue_entry_interval,40,21,0.238429


In [13]:
# Session-resolved within-interval cue effect maps (expert trials): data summary
def summarize_binwise_by_session(expert_bins_df, group_col, group_a, group_b, intervals=TARGET_INTERVALS, window_half_bins=1):
    rows = []
    for (session_id, interval), interval_df in expert_bins_df.groupby(['session_id', 'interval_name'], sort=True, dropna=False):
        if interval not in intervals:
            continue
        bin_ids = sorted(pd.Series(interval_df['bin_idx'].dropna().unique()).astype(int).tolist())
        for bin_idx in bin_ids:
            center = interval_df.loc[interval_df['bin_idx'] == bin_idx].copy()
            window = interval_df.loc[
                interval_df['bin_idx'].between(bin_idx - window_half_bins, bin_idx + window_half_bins)
            ].copy()
            stats_row = compare_two_groups(window, group_col, group_a, group_b, value_col='assembly_value')
            rows.append({
                'session_id': session_id,
                'interval_name': interval,
                'bin_idx': int(bin_idx),
                'bin_center_ms': float(center['bin_center_ms'].median()) if center['bin_center_ms'].notna().any() else np.nan,
                'window_half_bins': int(window_half_bins),
                'window_n_bins_used': int(window['bin_idx'].nunique()),
                **stats_row,
            })
    out = pd.DataFrame(rows)
    out['session_dt'] = pd.to_datetime(out['session_id'], format='%Y-%m-%d_%H-%M', errors='coerce')
    out = out.sort_values(['session_dt', 'session_id', 'interval_name', 'bin_idx']).reset_index(drop=True)
    session_order = (
        out[['session_id', 'session_dt']]
        .drop_duplicates()
        .sort_values(['session_dt', 'session_id'])
        .reset_index(drop=True)
    )
    session_order['session_order'] = np.arange(1, len(session_order) + 1)
    out = out.merge(session_order, on=['session_id', 'session_dt'], how='left')
    out = add_fdr_by_groups(out, ['session_id', 'interval_name'], p_col='welch_p_two_sided', out_col='welch_fdr_q')
    return out


session_bin_cue1_vs_cue2 = summarize_binwise_by_session(expert_bins_ens12, 'cue_label', 'cue1', 'cue2', window_half_bins=1)

print('Session-resolved binwise significant counts (expert trials, cue1 vs cue2; Welch FDR q < 0.05; center±1 bin window):')
display(
    session_bin_cue1_vs_cue2.groupby('interval_name', dropna=False)['welch_fdr_q']
    .apply(lambda s: int((s < 0.05).sum()))
    .reset_index(name='n_sig_session_bins')
)




Session-resolved binwise significant counts (expert trials, cue1 vs cue2; Welch FDR q < 0.05; center±1 bin window):


,interval_name,n_sig_session_bins
0,R1_entry_interval,31
1,R2_entry_interval,29
2,cue_entry_interval,18


In [ ]:
# session-resolved within-interval effect heatmap helper + expert cue plot


def plot_session_bin_effect_heatmaps_plotly(
    session_bin_df,
    title,
    diff_col='diff_b_minus_a',
    sig_col='welch_fdr_q',
    effect_label='effect',
    colorscale=None,
):
    if colorscale is None:
        # Negative -> orange, zero -> white, positive -> purple
        colorscale = [
            [0.0, '#f28e2b'],
            [0.5, '#f7f7f7'],
            [1.0, '#7b2cbf'],
        ]

    n_rows = len(TARGET_INTERVALS)
    fig = make_subplots(
        rows=n_rows,
        cols=1,
        shared_xaxes=False,
        vertical_spacing=0.10,
        subplot_titles=[i for i in TARGET_INTERVALS],
    )

    vmax_candidates = []
    for interval in TARGET_INTERVALS:
        sub = session_bin_df.loc[(session_bin_df['interval_name'] == interval) & session_bin_df[diff_col].notna()]
        vmax_candidates.append(np.nanpercentile(np.abs(sub[diff_col]), 95))
    vmax = max(vmax_candidates) if vmax_candidates else 1.0
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0

    for row_idx, interval in enumerate(TARGET_INTERVALS, start=1):
        sub = session_bin_df.loc[session_bin_df['interval_name'] == interval].copy()
        sub['bin_center_ms_plot'] = sub['bin_center_ms'].round(1)
        session_meta = (
            sub[['session_id', 'session_order']]
            .drop_duplicates()
            .sort_values(['session_order', 'session_id'])
            .reset_index(drop=True)
        )
        y_labels = session_meta['session_id'].tolist()

        heat = (
            sub.pivot_table(index='session_id', columns='bin_center_ms_plot', values=diff_col, aggfunc='mean')
            .reindex(index=y_labels)
            .sort_index(axis=1)
        )

        x_vals = heat.columns.to_numpy(dtype=float)
        y_vals = heat.index.tolist()
        z_vals = heat.to_numpy(dtype=float)

        fig.add_trace(
            go.Heatmap(
                x=x_vals,
                y=y_vals,
                z=z_vals,
                coloraxis='coloraxis',
                hovertemplate=(
                    'Interval: ' + interval + '<br>' +
                    'Time: %{x:.1f} ms<br>' +
                    'Session ID: %{y}<br>' +
                    effect_label + ': %{z:.4f}<extra></extra>'
                ),
            ),
            row=row_idx,
            col=1,
        )

        sig_sub = sub.loc[sub[sig_col] < 0.05].copy()
        sig_sub['bin_center_ms_plot'] = sig_sub['bin_center_ms'].round(1)
        sig_sub = sig_sub[
            sig_sub['bin_center_ms_plot'].isin(set(x_vals.tolist())) &
            sig_sub['session_id'].isin(set(y_vals))
        ]
        fig.add_trace(
            go.Scatter(
                x=sig_sub['bin_center_ms_plot'],
                y=sig_sub['session_id'],
                mode='markers',
                marker=dict(size=5, color='white', line=dict(color='rgba(0,0,0,0.35)', width=0.6)),
                name='Welch FDR q < 0.05' if row_idx == 1 else None,
                showlegend=(row_idx == 1),
                hovertemplate=(
                    'Interval: ' + interval + '<br>' +
                    'Time: %{x:.1f} ms<br>' +
                    'Session ID: %{y}<br>' +
                    'Marker: q < 0.05 (center±1 bin)<extra></extra>'
                ),
            ),
            row=row_idx,
            col=1,
        )

        fig.update_yaxes(
            title_text='Session ID',
            row=row_idx,
            col=1,
            type='category',
            categoryorder='array',
            categoryarray=y_labels,
            tickfont=dict(size=9),
            automargin=True,
        )
        fig.update_xaxes(
            title_text='Time from interval start (ms)' if row_idx == n_rows else None,
            showticklabels=(row_idx == n_rows),
            row=row_idx,
            col=1,
        )

    fig.update_layout(
        title=title,
        height=max(820, 300 * n_rows),
        width=1220,
        template='plotly_white',
        coloraxis=dict(
            colorscale=colorscale,
            cmin=-vmax,
            cmax=vmax,
            colorbar=dict(title=f'Effect size<br>({effect_label})'),
        ),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0),
        margin=dict(l=190, r=40, t=95, b=70),
    )
    return fig


fig_plotly_cue_heatmap = plot_session_bin_effect_heatmaps_plotly(
    session_bin_cue1_vs_cue2,
    title=f'{TARGET_ASSEMBLY}: session-resolved within-interval effect map (cue2 - cue1) [expert trials]',
    effect_label='cue2 - cue1',
)
fig_plotly_cue_heatmap.show()




# All Trials (including non-expert)

## Stop choice (R1 stop vs skip)
This section tests stop decision differences using all trials, avoiding the cue/action confound present in expert-only trials.



In [15]:
# All-trials stop decision: data prep (trial-level + within-interval)


def prepare_all_trial_interval_means(t0_ens, assembly_col=TARGET_ASSEMBLY, intervals=TARGET_INTERVALS):
    df = _analytics_df_with_columns(t0_ens)

    df = df[df['interval_name'].isin(intervals)].copy()
    df['cue'] = pd.to_numeric(df['cue'], errors='coerce')
    df = df[df[assembly_col].notna()].copy()
    df = df[df['choice_R1'].isin([True, False])].copy()

    df['cue_label'] = df['cue'].map({1.0: 'cue1', 2.0: 'cue2'})
    df['r1_action'] = np.where(df['choice_R1'].eq(True), 'stop_R1', 'skip_R1')

    group_cols = [
        'session_id', 'trial_id', 'interval_name', 'cue', 'cue_label',
        'choice_R1', 'choice_R2', 'r1_action'
    ]
    trial_means = (
        df.groupby(group_cols, dropna=False, as_index=False)[assembly_col]
        .mean()
        .rename(columns={assembly_col: 'assembly_mean'})
    )

    trial_means['session_dt'] = pd.to_datetime(
        trial_means['session_id'], format='%Y-%m-%d_%H-%M', errors='coerce'
    )
    session_order_df = (
        trial_means[['session_id', 'session_dt']]
        .drop_duplicates()
        .sort_values(['session_dt', 'session_id'], na_position='last')
        .reset_index(drop=True)
    )
    session_order_df['session_order'] = np.arange(1, len(session_order_df) + 1)
    trial_means = trial_means.merge(session_order_df, on=['session_id', 'session_dt'], how='left')
    return trial_means, session_order_df


def prepare_all_binwise_interval_data(t0_ens, assembly_col=TARGET_ASSEMBLY, intervals=TARGET_INTERVALS):
    df = _analytics_df_with_columns(t0_ens)
    needed = [
        'session_id', 'trial_id', 'cue', 'choice_R1', 'choice_R2', 'interval_name',
        'from_ephys_timestamp', 'to_ephys_timestamp', assembly_col
    ]
    optional = [c for c in ['entry_id', 't0_event_name'] if c in df.columns]
    df = df[needed + optional].copy()
    df = df[df['interval_name'].isin(intervals)].copy()
    df['cue'] = pd.to_numeric(df['cue'], errors='coerce')
    df = df[df[assembly_col].notna()].copy()
    df = df[df['choice_R1'].isin([True, False])].copy()

    df['cue_label'] = df['cue'].map({1.0: 'cue1', 2.0: 'cue2'})
    df['r1_action'] = np.where(df['choice_R1'].eq(True), 'stop_R1', 'skip_R1')
    df['assembly_value'] = df[assembly_col].astype(float)
    df['bin_width_us'] = (df['to_ephys_timestamp'] - df['from_ephys_timestamp']).astype(float)

    sort_cols = ['session_id', 'trial_id', 'interval_name', 'from_ephys_timestamp', 'to_ephys_timestamp']
    if 'entry_id' in df.columns:
        sort_cols.append('entry_id')
    df = df.sort_values(sort_cols).reset_index(drop=True)

    group_cols = ['session_id', 'trial_id', 'interval_name']
    df['bin_idx'] = df.groupby(group_cols).cumcount().astype(int)
    df['n_bins'] = df.groupby(group_cols)['assembly_value'].transform('size').astype(int)
    df['first_from_us'] = df.groupby(group_cols)['from_ephys_timestamp'].transform('min').astype(float)
    df['last_to_us'] = df.groupby(group_cols)['to_ephys_timestamp'].transform('max').astype(float)
    df['interval_duration_us'] = df['last_to_us'] - df['first_from_us']
    df['bin_center_us'] = (df['from_ephys_timestamp'].astype(float) + df['to_ephys_timestamp'].astype(float)) / 2.0
    df['bin_center_ms'] = (df['bin_center_us'] - df['first_from_us']) / 1000.0
    df['bin_center_frac'] = (df['bin_idx'] + 0.5) / df['n_bins'].replace(0, np.nan)

    interval_trial_meta = (
        df.groupby(group_cols, dropna=False)
        .agg(
            cue=('cue', 'first'),
            cue_label=('cue_label', 'first'),
            r1_action=('r1_action', 'first'),
            choice_R1=('choice_R1', 'first'),
            choice_R2=('choice_R2', 'first'),
            n_bins=('n_bins', 'first'),
            first_from_us=('from_ephys_timestamp', 'min'),
            last_to_us=('to_ephys_timestamp', 'max'),
            interval_duration_us=('interval_duration_us', 'first'),
        )
        .reset_index()
    )
    interval_trial_meta['session_dt'] = pd.to_datetime(interval_trial_meta['session_id'], format='%Y-%m-%d_%H-%M', errors='coerce')
    return df, interval_trial_meta


trial_means_all_ens12, session_order_all_ens12 = prepare_all_trial_interval_means(t0_ens)
all_bins_ens12, all_interval_trial_meta_ens12 = prepare_all_binwise_interval_data(t0_ens)

print(f'All-trials trial-level means rows: {len(trial_means_all_ens12)}')
print('All-trials counts per interval and stop decision (R1):')
print(
    trial_means_all_ens12.groupby(['interval_name', 'r1_action'])['trial_id']
    .count()
    .unstack(fill_value=0)
    .reindex(TARGET_INTERVALS, fill_value=0)
)




All-trials trial-level means rows: 8510
All-trials counts per interval and stop decision (R1):
r1_action           skip_R1  stop_R1
interval_name                       
R1_entry_interval      1857      980
R2_entry_interval      1857      979
cue_entry_interval     1857      980


In [ ]:
#All-trials stop decision: pooled and session-wise tests

overall_all_stop_vs_skip = summarize_overall(trial_means_all_ens12, 'r1_action', 'stop_R1', 'skip_R1')
session_all_stop_vs_skip = summarize_by_session(trial_means_all_ens12, 'r1_action', 'stop_R1', 'skip_R1')
trend_all_stop_vs_skip = summarize_trend(session_all_stop_vs_skip)

print('Overall comparison (all trials): stop vs skip (group_b - group_a = skip_R1 - stop_R1)')
display(overall_all_stop_vs_skip)
print()
print('Session-wise trend summary (all trials): stop vs skip (diff = skip_R1 - stop_R1)')
display(trend_all_stop_vs_skip)

plot_overall_group_comparison(
    trial_means_all_ens12,
    overall_all_stop_vs_skip,
    group_col='r1_action',
    group_order=['stop_R1', 'skip_R1'],
    title=f'{TARGET_ASSEMBLY}: all trials, stop vs skip (pooled across sessions)',
    colors=['#f28e2b', '#7b2cbf'],
)

plot_session_differences(
    session_all_stop_vs_skip,
    title=f'{TARGET_ASSEMBLY}: session-wise difference over time (skip_R1 - stop_R1, all trials)',
    y_label='Mean diff (group_b - group_a)',
)



Overall comparison (all trials): stop vs skip (group_b - group_a = skip_R1 - stop_R1)


,interval_name,group_a,group_b,n_a,n_b,mean_a,mean_b,std_a,std_b,diff_b_minus_a,welch_t,welch_p_two_sided,welch_p_one_sided_b_gt_a,mw_u,mw_p_two_sided
0,R1_entry_interval,stop_R1,skip_R1,980,1857,1.272700,1.141329,0.994120,0.881450,-0.131371,3.477877,5.174396e-04,2.587198e-04,971955.0,0.002792
1,R2_entry_interval,stop_R1,skip_R1,979,1857,1.400968,1.197778,1.056536,0.882060,-0.203190,5.145810,2.971054e-07,1.485527e-07,1001116.0,0.000009
2,cue_entry_interval,stop_R1,skip_R1,980,1857,1.134866,0.982259,1.000591,0.865977,-0.152607,4.042018,5.527664e-05,2.763832e-05,981281.0,0.000583



Session-wise trend summary (all trials): stop vs skip (diff = skip_R1 - stop_R1)


,interval_name,n_sessions,spearman_rho,spearman_p
0,R1_entry_interval,28,-0.212370,0.277941
1,R2_entry_interval,28,0.205255,0.294733
2,cue_entry_interval,28,-0.013684,0.944903


In [ ]:
# All-trials stop decision: trial-index + within-interval progression + heatmap

trial_index_all_stop_vs_skip = summarize_by_trial_index(trial_means_all_ens12, 'r1_action', 'stop_R1', 'skip_R1')
trial_index_trend_all_stop_vs_skip = summarize_trial_index_trend(trial_index_all_stop_vs_skip)
trial_traj_all_stop_vs_skip = summarize_trial_index_group_trajectory(
    trial_means_all_ens12,
    group_col='r1_action',
    group_order=['stop_R1', 'skip_R1'],
)

print('Trial-index trend summary (all trials): stop vs skip (diff = skip_R1 - stop_R1)')
display(trial_index_trend_all_stop_vs_skip)
print()
print('Significant trial_ids (all trials, stop vs skip; Welch FDR q < 0.05):')
display(trial_index_all_stop_vs_skip.loc[trial_index_all_stop_vs_skip['welch_fdr_q'] < 0.05].head(20))
print('... total significant rows:', int((trial_index_all_stop_vs_skip['welch_fdr_q'] < 0.05).sum()))

plot_trial_index_differences(
    trial_index_all_stop_vs_skip,
    title=f'{TARGET_ASSEMBLY}: trial-number based difference (skip_R1 - stop_R1, all trials, pooled across sessions)',
)

plot_trial_index_group_trajectories(
    trial_traj_all_stop_vs_skip,
    trial_index_all_stop_vs_skip,
    group_col='r1_action',
    group_order=['stop_R1', 'skip_R1'],
    title=f'{TARGET_ASSEMBLY}: trial-number mean trajectories (stop vs skip, all trials)',
    colors=['#f28e2b', '#7b2cbf'],
)

binwise_all_stop_vs_skip_raw = summarize_binwise_overall(all_bins_ens12, 'r1_action', 'stop_R1', 'skip_R1', window_half_bins=0)
binwise_all_stop_vs_skip = summarize_binwise_overall(all_bins_ens12, 'r1_action', 'stop_R1', 'skip_R1', window_half_bins=1)
bintraj_all_stop_vs_skip = summarize_binwise_group_trajectory(all_bins_ens12, 'r1_action', ['stop_R1', 'skip_R1'])
session_bin_all_stop_vs_skip = summarize_binwise_by_session(all_bins_ens12, 'r1_action', 'stop_R1', 'skip_R1')

print()
print('Within-interval binwise significance summary (all trials, stop vs skip):')
display(
    binwise_all_stop_vs_skip.groupby('interval_name', dropna=False)
    .agg(
        n_bins_tested=('bin_idx', 'size'),
        n_sig_welch_fdr=('welch_fdr_q', lambda s: int((s < 0.05).sum())),
        median_abs_diff=('diff_b_minus_a', lambda s: float(np.nanmedian(np.abs(s)))),
    )
    .reset_index()
)

print()
print('Session-resolved binwise significant counts (all trials, stop vs skip; Welch FDR q < 0.05):')
display(
    session_bin_all_stop_vs_skip.groupby('interval_name', dropna=False)['welch_fdr_q']
    .apply(lambda s: int((s < 0.05).sum()))
    .reset_index(name='n_sig_session_bins')
)

plot_binwise_trajectories_with_significance(
    bintraj_all_stop_vs_skip,
    [
        {
            'summary_df': binwise_all_stop_vs_skip_raw,
            'panel_title': 'Raw bins',
            'sig_label': 'q < 0.05 (raw bin)',
            'window_desc': 'center bin only',
        },
        {
            'summary_df': binwise_all_stop_vs_skip,
            'panel_title': '3-bin window',
            'sig_label': 'q < 0.05 (3-bin window)',
            'window_desc': 'center±1 bin',
        },
    ],
    group_col='r1_action',
    group_order=['stop_R1', 'skip_R1'],
    title=f'{TARGET_ASSEMBLY}: within-interval progression (stop vs skip, all trials) | raw-bin vs center±1-bin significance',
    colors=['#f28e2b', '#7b2cbf'],
)

fig_plotly_all_stop_heatmap = plot_session_bin_effect_heatmaps_plotly(
    session_bin_all_stop_vs_skip,
    title=f'{TARGET_ASSEMBLY}: session-resolved within-interval effect map (skip_R1 - stop_R1) [all trials]',
    effect_label='skip_R1 - stop_R1',
)
fig_plotly_all_stop_heatmap.show()



Trial-index trend summary (all trials): stop vs skip (diff = skip_R1 - stop_R1)


,interval_name,n_trial_ids,spearman_rho,spearman_p
0,R1_entry_interval,162,0.268622,5.477044e-04
1,R2_entry_interval,162,0.170316,3.024748e-02
2,cue_entry_interval,162,0.524803,7.575380e-13



Significant trial_ids (all trials, stop vs skip; Welch FDR q < 0.05):


,trial_id,interval_name,group_a,group_b,n_sessions_a,n_sessions_b,n_a,n_b,mean_a,mean_b,std_a,std_b,diff_b_minus_a,welch_t,welch_p_two_sided,welch_p_one_sided_b_gt_a,mw_u,mw_p_two_sided,welch_fdr_q,mw_fdr_q
482,93,cue_entry_interval,stop_R1,skip_R1,4,13,4,13,0.124002,1.344512,0.299377,0.658014,1.22051,-5.170864,0.000236,0.999882,1.0,0.001681,0.031404,0.136134


... total significant rows: 1



Within-interval binwise significance summary (all trials, stop vs skip):


,interval_name,n_bins_tested,n_sig_welch_fdr,median_abs_diff
0,R1_entry_interval,45,16,0.112570
1,R2_entry_interval,40,31,0.201583
2,cue_entry_interval,40,24,0.145287



Session-resolved binwise significant counts (all trials, stop vs skip; Welch FDR q < 0.05):


,interval_name,n_sig_session_bins
0,R1_entry_interval,82
1,R2_entry_interval,30
2,cue_entry_interval,16
